# [업무용] 휴가 계획 도우미 — 회사 휴가 규정 + 실제 공휴일

**「코랩으로 만드는 업무 챗봇」 부록 · 챗봇 콘테스트 과제 1 「내 일을 덜어 주는 챗봇」 모범 예제** — 과제 조건 다섯 가지를 모두 채운 완성형 예시입니다. 그대로 돌려 본 뒤 맨 아래 안내대로 내 주제로 바꿔 보세요.

### 이 챗봇이 덜어 주는 일
휴가를 잡으려면 ① 올해 공휴일이 무슨 요일인지 달력을 넘겨 보고 ② 하루만 쉬면 길게 이어지는 날을 찾고 ③ 며칠 전까지 신청해야 하는지 규정집을 뒤집니다. 팀장과 인사 담당자는 같은 질문을 여러 사람에게서 받습니다.
이 챗봇은 **공휴일 데이터(외부 API)** 와 **회사 휴가 규정(내 문서)** 을 함께 찾아, 근거와 데이터를 받은 시각을 붙여 답합니다.

### 시작하는 방법
1. 「Drive로 복사」 → 메뉴 **런타임 → 모두 실행**. 맨 아래 칸에 챗봇 화면과 `https://….gradio.live` 주소가 뜹니다(2~3분).
2. **키 없이 끝까지 돕니다.** 검색은 코랩 안에서 도는 임베딩 모델로 하고, 답은 찾은 조각을 그대로 보여 줍니다.
3. 코랩 왼쪽 🔑 보안 비밀에 `OPENAI_API_KEY` 를 넣으면 답을 문장으로 다듬어 줍니다(권한 창이 뜨면 「액세스 권한 부여」).

### 과제 조건과 칸 대응
| 필수 조건 | 이 노트북에서 | 심사 항목 |
|---|---|---|
| 코랩 노트북 하나에서 설치부터 실행까지 | ① 설치 ~ ⑧ 화면 | 완성도 · 사용성 20 |
| 외부 API 1개 이상으로 실제 데이터 | ③ 공휴일 API(Nager.Date · 키 없음), 실패하면 예시 데이터로 바꾸고 화면에 표시 | 외부 API 활용 20 |
| RAG 구조(청킹 → 임베딩 → 검색) | ④ 청킹 · ⑤ 임베딩 검색 | **RAG · 청킹 설계 25** |
| 청크 크기 · 겹침을 2가지 이상 비교 | ⑤ 세 가지 설정 × 같은 질문 3개 비교표 + 이유 | 〃 |
| Gradio 화면, 답에 근거(출처 · 데이터 시각) | ⑧ 화면 | 완성도 · 사용성 20 |
| 키는 코드에 쓰지 않고 보안 비밀에 | ② 키 불러오기 | — |
| (심사) 모르면 모른다고 하는가 | ⑥ 기준 점수 | 신뢰성 15 |
| (심사) 실제로 쓸 사람이 있는가 | 위 「덜어 주는 일」 · ⑦ 날짜 계산 | 문제 정의 · 쓸모 20 |

## ① 설치 (1분쯤)

In [ ]:
import importlib, importlib.util, os, subprocess, sys, time

os.environ["TZ"] = "Asia/Seoul"            # 코랩 시계는 UTC — 「받은 시각」을 한국 기준으로
time.tzset()

NEED = {"fastembed": "fastembed", "gradio": "gradio", "openai": "openai", "pandas": "pandas",
        "langchain_text_splitters": "langchain-text-splitters", "langchain_core": "langchain-core"}
for mod, pkg in NEED.items():
    if importlib.util.find_spec(mod) is None:
        print("설치:", pkg)
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", pkg], check=False, capture_output=True)
importlib.invalidate_caches()
missing = [m for m in NEED if importlib.util.find_spec(m) is None]
print("아직 없음 → " + ", ".join(missing) + " · 이 칸을 한 번 더 실행해 주세요." if missing else "설치 확인 완료")

## ② 키 불러오기 (없어도 됩니다)
키는 코드에 적지 않습니다. 코랩 왼쪽 🔑 **보안 비밀**에 넣은 값을 읽어 옵니다.

In [ ]:
try:
    from google.colab import userdata           # 코랩 보안 비밀
    os.environ["OPENAI_API_KEY"] = userdata.get("OPENAI_API_KEY") or ""
except Exception:
    pass
HAS_KEY = bool(os.environ.get("OPENAI_API_KEY"))
print("OPENAI_API_KEY", "있음 — 답을 문장으로 다듬습니다" if HAS_KEY else "없음 — 찾은 조각을 그대로 보여 줍니다")

## ③ 데이터 준비 — 내 문서 1개 + 외부 API 1개
- **내 문서**: 회사 휴가 규정입니다. 여기서는 실습용으로 만든 가상의 규정을 씁니다. 내 작품에서는 실제 사내 문서로 바꿉니다.
- **외부 API**: [Nager.Date](https://date.nager.at) 의 나라별 공휴일 API 입니다. 키 없이 부를 수 있습니다. 올해와 내년 치를 받아 **날짜마다 한 문단**으로 정리합니다.

API 응답(JSON)을 그대로 넣지 않고 「2026-10-09 (금) 한글날」처럼 **사람이 읽는 문장으로 바꿔** 넣습니다. 요일은 응답에 없어서 파이썬이 계산해 붙입니다.
API 를 부르지 못하면 **예시 데이터**로 대신하고 출처에 그렇게 적습니다 — 실제 값이 아니라는 것이 화면에도 보입니다.

In [ ]:
RULES = """# 휴가 운영 규정 (실습용으로 만든 가상의 회사 규정입니다)

## 제1조 목적
이 규정은 구성원이 휴가를 미리 계획해 쉬고, 팀이 업무 공백 없이 일정을 짤 수 있게 하려고 만든다. 이 규정에 없는 사항은 인사팀에 문의한다.

## 제2조 연차 일수
입사 1년 미만인 구성원은 한 달을 모두 근무하면 연차 1일이 생긴다. 입사 1년 이상인 구성원은 해마다 연차 15일을 받는다. 3년 이상 근무하면 그 뒤 2년마다 1일씩 더해 최대 25일까지 받는다.

## 제3조 신청 절차
연차는 쓰려는 날의 3일 전까지 근태 시스템에 신청한다. 연속해서 5일 이상 쓰는 연차는 2주 전까지 신청하고 팀장의 승인을 받는다. 급한 사정으로 당일에 신청할 때는 오전 9시 전에 팀장에게 전화로 먼저 알린다.

## 제4조 반차와 반반차
반차는 4시간, 반반차는 2시간 단위로 쓴다. 오전 반차는 오후 2시에 출근하고 오후 반차는 오후 2시에 퇴근한다. 반반차는 한 달에 4번까지 쓸 수 있다.

## 제5조 징검다리 권장 휴가
공휴일과 주말 사이에 낀 평일 하루는 징검다리 권장 휴가일로 본다. 이날 연차를 신청하면 팀장은 특별한 사정이 없으면 먼저 승인한다. 인사팀은 해마다 1월에 그해의 징검다리 권장 휴가일을 공지한다.

## 제6조 경조 휴가
본인이 결혼하면 5일, 자녀가 결혼하면 1일의 경조 휴가를 준다. 배우자가 출산하면 10일을 준다. 부모나 배우자가 사망하면 5일을 준다. 경조 휴가는 연차에서 빼지 않는다.

## 제7조 이월과 소멸
쓰지 못한 연차는 다음 해 3월 31일까지 쓸 수 있고 그 뒤에는 없어진다. 이월할 수 있는 연차는 최대 5일이다. 인사팀은 해마다 10월에 남은 연차 일수를 구성원에게 알린다.

## 제8조 병가
아파서 일하기 어려우면 한 해에 30일까지 병가를 쓸 수 있다. 연속해서 3일 이상 쓰는 병가는 진단서를 낸다. 병가 가운데 처음 10일은 유급이다.
"""

import datetime as dt
import json, urllib.request

WEEK = "월화수목금토일"
SAMPLE = [("2026-01-01", "새해"), ("2026-10-05", "개천절"), ("2026-10-09", "한글날"), ("2026-12-25", "크리스마스")]


def now():
    return dt.datetime.now().strftime("%Y-%m-%d %H:%M")


def fetch_holidays(year):
    """공휴일 API → [(날짜, 이름)]"""
    url = f"https://date.nager.at/api/v3/PublicHolidays/{year}/KR"
    with urllib.request.urlopen(url, timeout=15) as r:
        return [(x["date"], x["localName"]) for x in json.load(r)]


def holiday_text(year, rows):
    lines = [f"# {year}년 공휴일"]
    for date, name in rows:
        d = dt.date.fromisoformat(date)
        lines.append(f"## {date} ({WEEK[d.weekday()]}) {name}\n{year}년 {name}은(는) {d.month}월 {d.day}일 {WEEK[d.weekday()]}요일이다.")
    return "\n\n".join(lines)


def load_docs(year=None):
    """문서 목록 = 규정 1개 + 공휴일(올해 · 내년). 각 문서에 출처와 받은 시각을 붙입니다."""
    year = int(year or dt.date.today().year)
    docs = [{"text": RULES, "source": "회사 휴가 운영 규정(내 문서)", "fetched_at": "노트북에 포함"}]
    holidays = {}
    for y in (year, year + 1):
        try:
            rows = fetch_holidays(y)
            src = f"Nager.Date 공휴일 API · {y}년"
        except Exception as e:
            print(f"API 를 부르지 못해 {y}년은 예시 데이터로 대신합니다 →", e)
            rows = [(d.replace("2026", str(y)), n) for d, n in SAMPLE]
            src = f"예시 데이터(실제 값 아님) · {y}년"
        holidays.update(rows)
        docs.append({"text": holiday_text(y, rows), "source": src, "fetched_at": now()})
    return docs, holidays


docs, HOLIDAYS = load_docs()
for d in docs:
    print(f"{d['source']:34s} {d['fetched_at']:18s} {len(d['text']):>5,}자")
print()
print(docs[1]["text"][:230], "…")

## ④ 청킹 — 크기와 겹침을 세 가지로
같은 문서를 세 가지 설정으로 자릅니다. 분할기는 6장의 재귀 분할기(`RecursiveCharacterTextSplitter`)이고, `##` 제목 → 빈 줄 → 줄 → 문장 순서로 자를 자리를 찾습니다.

| 설정 | 뜻 | 예상 |
|---|---|---|
| 120 / 0 | 아주 잘게, 겹침 없이 | 조항이 잘게 갈라지고 제목만 남은 조각이 생긴다 |
| 200 / 30 | 조항 하나쯤 | 조항 하나가 조각 하나에 들어간다 |
| 1200 / 150 | 크게 | 여러 조항 · 여러 공휴일이 한 조각에 섞인다 |

In [ ]:
import re, statistics
import pandas as pd
from langchain_core.documents import Document
from langchain_text_splitters import RecursiveCharacterTextSplitter

SETTINGS = [(120, 0), (200, 30), (1200, 150)]            # (chunk_size, chunk_overlap)
END = re.compile(r"([.!?)\]]|다|요)\s*$")                 # 이렇게 끝나지 않으면 문장 중간에서 잘린 조각


def split(docs, size, overlap):
    sp = RecursiveCharacterTextSplitter(chunk_size=size, chunk_overlap=overlap,
                                        separators=["\n## ", "\n\n", "\n", ". ", " ", ""])
    return sp.split_documents([Document(page_content=d["text"], metadata={"source": d["source"], "fetched_at": d["fetched_at"]})
                               for d in docs])


CHUNKS = {s: split(docs, *s) for s in SETTINGS}
rows = []
for (size, ov), chunks in CHUNKS.items():
    L = [len(c.page_content) for c in chunks]
    rows.append({"설정": f"{size} / {ov}", "조각 수": len(L), "평균 길이": round(statistics.mean(L)), "최대 길이": max(L),
                 "40자 미만 조각": sum(1 for n in L if n < 40),
                 "문장 중간 절단": sum(1 for c in chunks if not END.search(c.page_content.strip()))})
print(pd.DataFrame(rows).to_string(index=False))

print("\n── 40자 미만 조각(120 / 0) →", [c.page_content for c in CHUNKS[(120, 0)] if len(c.page_content) < 40])
print("\n── 「제3조 신청 절차」가 설정마다 어떻게 잘렸나")
for s, chunks in CHUNKS.items():
    part = [c.page_content for c in chunks if "2주 전" in c.page_content][0]
    print(f"\n[{s[0]} / {s[1]}] {len(part)}자\n{part[:260]}{' …' if len(part) > 260 else ''}")

## ⑤ 임베딩 · 검색 — 같은 질문 3개로 설정을 비교합니다
조각을 임베딩해 질문과 가까운 순서로 찾습니다. 임베딩 모델은 코랩 안에서 도는 다국어 모델이라 키가 필요 없습니다(첫 실행은 내려받느라 30초쯤).

비교표의 칸은 이렇게 읽습니다.
- **정답 순위**: 답이 든 조각(핵심어 포함)이 검색 결과 몇 위였나. ✗ = 3위 안에 없음
- **1위 점수**: 1위 조각과 질문의 유사도
- **보내는 글자**: 상위 3조각의 글자 수. AI 에게 넘기는 양이라 클수록 비용과 잡음이 늘어납니다
- **문서 안 최저 · 문서 밖 최고**: 답이 있는 질문 6개의 1위 점수 중 가장 낮은 값과, 관계없는 질문 4개의 1위 점수 중 가장 높은 값. **안 최저가 밖 최고보다 높아야** 「모르면 모른다」를 가를 기준을 세울 수 있습니다
- 표 아래에는 **같은 질문에 1위로 나온 조각**을 설정별로 나란히 보여 줍니다 — 답이 어떻게 달라지는지 여기서 봅니다

In [ ]:
import numpy as np
from fastembed import TextEmbedding

_model = TextEmbedding("sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2", cache_dir="/content/fastembed")


def embed(texts):
    M = np.array(list(_model.embed(list(texts))))
    return M / np.linalg.norm(M, axis=1, keepdims=True)


class Index:
    def __init__(self, chunks):
        self.chunks, self.M = chunks, embed([c.page_content for c in chunks])

    def search(self, q, k=3):
        s = self.M @ embed([q])[0]
        return [(self.chunks[i], round(float(s[i]), 3)) for i in np.argsort(-s)[:k]]


# (질문, 답이 든 조각이라면 들어 있어야 하는 핵심어)
QUESTIONS = [("5일 이상 이어서 쉬려면 언제까지 신청해야 하나요?", "2주 전"),
             ("쓰지 못한 연차는 언제 없어지나요?", "3월 31일"),
             ("배우자가 출산하면 휴가가 며칠인가요?", "10일")]

INSIDE = [q for q, _ in QUESTIONS] + ["반반차는 한 달에 몇 번 쓸 수 있나요?", "한글날은 무슨 요일인가요?", "병가는 며칠까지 쓸 수 있나요?"]
OUTSIDE = ["김치찌개 끓이는 법 알려줘", "오늘 서울 날씨 어때?", "축구 경기는 몇 명이 뛰나요?", "비트코인 지금 얼마야?"]

INDEX = {s: Index(chunks) for s, chunks in CHUNKS.items()}
rows, firsts = [], {}
for s, idx in INDEX.items():
    r = {"설정": f"{s[0]} / {s[1]}", "조각 수": len(idx.chunks)}
    hit, sent = 0, 0
    for i, (q, kw) in enumerate(QUESTIONS, 1):
        top = idx.search(q, 3)
        rank = next((n for n, (c, _) in enumerate(top, 1) if kw in c.page_content), None)
        r[f"질문{i} 정답 순위"] = f"{rank}위" if rank else "✗"
        hit += rank == 1
        sent += sum(len(c.page_content) for c, _ in top)
        firsts[(s, i)] = top[0][0].page_content
    r["1위 적중"] = f"{hit}/{len(QUESTIONS)}"
    r["보내는 글자"] = sent // len(QUESTIONS)
    r["문서 안 최저"] = min(idx.search(q, 1)[0][1] for q in INSIDE)
    r["문서 밖 최고"] = max(idx.search(q, 1)[0][1] for q in OUTSIDE)
    r["갈리는가"] = "예" if r["문서 안 최저"] > r["문서 밖 최고"] else "아니오"
    rows.append(r)
table = pd.DataFrame(rows)
print(table.to_string(index=False))

for i, (q, kw) in enumerate(QUESTIONS, 1):
    print(f"\n── 질문{i}: {q}  (핵심어 「{kw}」)")
    for s in SETTINGS:
        t = firsts[(s, i)].replace("\n", " ")
        print(f"   [{s[0]:>4} / {s[1]:<3}] {'○' if kw in t else '✗'} {len(t):>4}자 │ {t[:74]}{'…' if len(t) > 74 else ''}")

bad = max(OUTSIDE, key=lambda q: INDEX[(120, 0)].search(q, 1)[0][1])
c, sc = INDEX[(120, 0)].search(bad, 1)[0]
print(f"\n── 120 / 0 에서 「{bad}」의 1위 조각 → 점수 {sc} │ {c.page_content!r}")

# 고르는 규칙: ① 문서 안 · 밖 질문이 갈리는 설정 중에서 ② 1위 적중이 많고 ③ 보내는 글자가 적은 것
best = max(rows, key=lambda r: (r["갈리는가"] == "예", int(r["1위 적중"][0]), -r["보내는 글자"]))
BEST = tuple(int(x) for x in best["설정"].split(" / "))
print(f"\n→ 고른 설정 {BEST[0]} / {BEST[1]} — 갈리는가 {best['갈리는가']} · 1위 적중 {best['1위 적중']} · 질문당 {best['보내는 글자']:,}자")

### 왜 200 / 30 으로 정했나 — 같은 질문에서 답이 이렇게 달라졌습니다

| 설정 | 무슨 일이 생겼나 | 판단 |
|---|---|---|
| **120 / 0** | 세 질문 모두 1위로 맞혔고 보내는 글자도 가장 적습니다(307자). 그런데 `## 제2조 연차 일수` 처럼 **제목만 남은 조각이 4개** 생겼고, 「김치찌개 끓이는 법 알려줘」가 그 조각에 **0.74** 로 걸렸습니다. 답이 있는 질문의 최저 점수(0.504)보다 높아서 기준 점수를 어디에 둬도 못 거릅니다. | 적중률만 보면 1등, 신뢰성에서 탈락 |
| **200 / 30** | 세 질문 모두 1위(3/3), 보내는 글자 368자. 조항 하나가 조각 하나에 들어가 제목과 본문이 함께 있습니다. 문서 안 최저 0.355 · 문서 밖 최고 0.277 로 **갈립니다**. | **선택** |
| **1200 / 150** | 규정 전체가 조각 하나(944자)가 되어 어떤 질문에도 같은 덩어리가 나옵니다. 「배우자 출산」 질문에는 1위로 **2027년 공휴일 목록**이 나왔습니다. 보내는 글자는 2,754자로 7배이고, 문서 안 최저(0.249)가 문서 밖 최고(0.292)보다 낮아 기준도 세울 수 없습니다. | 탈락 |

- **크기를 200 으로 한 이유**: 이 규정은 조항 하나가 100~200자입니다. 조각 하나에 조항 하나가 들어가는 크기라, 질문 하나에 조항 하나가 답으로 나옵니다. 문서의 「한 덩어리」가 얼마인지 먼저 재고 크기를 거기에 맞췄습니다.
- **겹침을 30 으로 한 이유**: 크기의 15%입니다. 120 / 0 처럼 겹침이 없으면 제목과 본문이 다른 조각으로 떨어졌을 때 이어 줄 것이 없습니다.
- **적중률만으로 고르지 않은 이유**: 「잘 찾는가」와 「모를 때 모른다고 하는가」는 따로 재야 합니다. 120 / 0 은 앞의 것만 좋았습니다.

위 숫자는 이 노트북을 만든 날 실측한 값입니다. 공휴일 데이터가 바뀌면 조금 달라질 수 있으니, 내 작품에서는 **내 표의 값**으로 다시 적습니다.

## ⑥ 기준 점수 — 모르면 모른다고 답하기
검색은 무엇을 물어도 「가장 가까운 조각」을 내놓습니다. 관계없는 질문에 엉뚱한 근거를 붙이지 않으려면 **1위 점수가 기준보다 낮을 때 답하지 않게** 해야 합니다.
고른 설정에서 문서 안 질문과 문서 밖 질문의 1위 점수를 펼쳐 보고, 두 무리 사이 한가운데에 기준을 둡니다.

In [ ]:
KB = INDEX[BEST]                                          # 고른 설정의 검색기

rows = [{"종류": kind, "질문": q, "1위 점수": KB.search(q, 1)[0][1]} for kind, qs in (("문서 안", INSIDE), ("문서 밖", OUTSIDE)) for q in qs]
t = pd.DataFrame(rows)
print(t.to_string(index=False))
low_in, high_out = t[t["종류"] == "문서 안"]["1위 점수"].min(), t[t["종류"] == "문서 밖"]["1위 점수"].max()
THRESHOLD = round((low_in + high_out) / 2, 2) if low_in > high_out else round(low_in, 2)
print(f"\n문서 안 최저 {low_in} · 문서 밖 최고 {high_out} → 기준 {THRESHOLD}" + ("" if low_in > high_out else "  (두 값이 겹칩니다 — 질문을 바꿔 다시 재 보세요)"))

## ⑦ 날짜 계산은 파이썬이 합니다
「징검다리 연휴가 언제야?」는 검색으로 풀 문제가 아니라 **계산**입니다. 공휴일과 주말 사이에 낀 평일 하루를 파이썬이 달력으로 찾아 [계산] 자료로 붙입니다. AI 에게 날짜를 세게 하면 틀립니다.
규정 제5조(징검다리 권장 휴가)와 이 계산이 합쳐져야 「이날 쉬면 며칠 이어지고, 신청하면 먼저 승인된다」는 쓸모 있는 답이 됩니다.

In [ ]:
def is_off(d, holidays):
    return d.weekday() >= 5 or d.isoformat() in holidays


def bridges(holidays, start=None):
    """오늘 이후, 쉬는 날 사이에 낀 평일 하루 → [(날짜, 그날 쉬면 이어지는 일수, 사이에 낀 공휴일 이름)]"""
    start = start or dt.date.today()
    out, last = [], max(dt.date.fromisoformat(x) for x in holidays)
    d = start
    while d <= last:
        one = dt.timedelta(days=1)
        if not is_off(d, holidays) and is_off(d - one, holidays) and is_off(d + one, holidays):
            a, b = d, d
            while is_off(a - one, holidays):
                a -= one
            while is_off(b + one, holidays):
                b += one
            names = [holidays[x.isoformat()] for x in (a + dt.timedelta(n) for n in range((b - a).days + 1)) if x.isoformat() in holidays]
            if names:                                   # 주말만 이어진 날은 뺍니다
                out.append((d, (b - a).days + 1, a, b, " · ".join(dict.fromkeys(names))))
        d += one
    return out


def calc_text(holidays):
    today = dt.date.today()
    nxt = sorted(x for x in holidays if x >= today.isoformat())[:3]
    lines = [f"[계산] 기준일 {today} ({WEEK[today.weekday()]})",
             "다가오는 공휴일: " + ", ".join(f"{x}({WEEK[dt.date.fromisoformat(x).weekday()]}) {holidays[x]}" for x in nxt)]
    for d, n, a, b, names in bridges(holidays)[:6]:
        lines.append(f"징검다리: {d}({WEEK[d.weekday()]}) 하루 연차를 쓰면 {a}~{b} {n}일 연속 휴식 ({names})")
    return "\n".join(lines)


print(calc_text(HOLIDAYS))

## ⑧ 완성 — 챗봇 화면
답의 순서는 **찾기 → 기준 확인 → (날짜 질문이면 계산 붙이기) → 답 → 근거**입니다. 근거에는 출처와 데이터를 받은 시각, 유사도 점수가 붙습니다.
「데이터 새로 받기」를 누르면 공휴일 API 를 다시 부릅니다. 멈추려면 `demo.close()` 를 실행합니다.

In [ ]:
import gradio as gr

DATE_WORDS = ("징검다리", "연휴", "공휴일", "언제 쉬", "쉬는 날", "황금")
STATE = {"kb": KB, "holidays": HOLIDAYS, "docs": docs}
SYSTEM = """너는 회사의 휴가 계획을 돕는 도우미야. 아래 [자료]만 근거로 한국어로 짧고 쉽게 답해.
- 날짜 · 일수 · 요일은 자료에 적힌 그대로 써. 자료에 [계산] 이 있으면 그 값을 쓰고 직접 날짜를 세지 마.
- 자료에 없는 내용은 「자료에서 찾지 못했어요」라고 말하고 지어내지 마."""


def respond(q):
    q = (q or "").strip()
    hits = STATE["kb"].search(q, 3)
    dated = any(w in q for w in DATE_WORDS)
    if hits[0][1] < THRESHOLD and not dated:              # 근거가 없으면 여기서 끝 — AI 를 부르지 않습니다
        return f"자료에서 찾지 못했어요. (가장 가까운 조각의 점수 {hits[0][1]} < 기준 {THRESHOLD})\n휴가 규정과 공휴일에 대해 물어봐 주세요."
    used = [(c, s) for c, s in hits if s >= THRESHOLD][:2] or hits[:1]
    extra = calc_text(STATE["holidays"]) if dated else ""
    ctx = "\n\n".join(c.page_content for c, _ in used) + ("\n\n" + extra if extra else "")
    if HAS_KEY:
        from openai import OpenAI
        try:
            ans = OpenAI().chat.completions.create(model="gpt-4o-mini", temperature=0, messages=[
                {"role": "system", "content": SYSTEM}, {"role": "user", "content": f"[자료]\n{ctx}\n\n[질문]\n{q}"}]).choices[0].message.content
        except Exception as e:
            ans = f"(AI 답을 받지 못해 찾은 자료를 그대로 보여 드립니다 — {str(e)[:80]})\n\n{ctx}"
    else:
        ans = extra if extra else used[0][0].page_content
    refs = "\n".join(f"- {c.metadata['source']} · 받은 시각 {c.metadata['fetched_at']} · 점수 {s}" for c, s in used)
    return f"{ans}\n\n---\n**근거**\n{refs}" + ("\n- 날짜 계산: 파이썬(공휴일 데이터 기준)" if extra else "")


def chat(message, history):
    return respond(message)


def refresh(year):
    d, h = load_docs(year)
    STATE.update(docs=d, holidays=h, kb=Index(split(d, *BEST)))
    src = d[1]
    if src["source"].startswith("예시"):
        gr.Warning("공휴일 API 를 부르지 못해 예시 데이터로 대신합니다. 날짜는 실제 값이 아닙니다.")
        return f"새로 받지 못했습니다 — {src['source']} · {src['fetched_at']}"
    return f"새로 받았습니다 — {src['source']} 외 · {src['fetched_at']} · 조각 {len(STATE['kb'].chunks)}개"


with gr.Blocks(title="휴가 계획 도우미") as demo:
    gr.Markdown("# 휴가 계획 도우미\n회사 휴가 규정과 실제 공휴일 데이터를 찾아 근거와 함께 답합니다. 자료에 없으면 모른다고 답합니다.")
    with gr.Row():
        year = gr.Textbox(label="기준 연도", value=str(dt.date.today().year), scale=1)
        btn = gr.Button("데이터 새로 받기", scale=1)
        status = gr.Markdown(f"데이터: {docs[1]['source']} 외 · {docs[1]['fetched_at']} · 청크 {BEST[0]}/{BEST[1]} · 기준 {THRESHOLD}")
    gr.ChatInterface(chat, examples=["다가오는 징검다리 연휴는 언제인가요?", "5일 이상 이어서 쉬려면 언제까지 신청해야 하나요?",
                                     "쓰지 못한 연차는 언제 없어지나요?", "김치찌개 끓이는 법 알려줘"])
    btn.click(refresh, year, status)

for q in ["다가오는 징검다리 연휴는 언제인가요?", "쓰지 못한 연차는 언제 없어지나요?", "김치찌개 끓이는 법 알려줘"]:
    print("질문:", q, "\n" + respond(q), "\n")

demo.launch(share="google.colab" in sys.modules)

## 내 작품으로 바꾸려면
1. **③ 칸** — `RULES` 를 내 문서로, `fetch_holidays` 를 내가 쓸 API 로 바꿉니다. API 응답은 사람이 읽는 문장으로 바꿔 넣고, 출처와 받은 시각을 꼭 붙입니다.
2. **⑤ 칸** — `QUESTIONS` 의 질문 3개와 핵심어를 내 주제로 바꿉니다. 비교표가 이 질문으로 채워집니다.
3. **④ · ⑤ 칸**을 다시 실행해 설정을 고르고, 「왜 이 크기로 정했는가」를 **내 말로** 적습니다(심사 25점).
4. **⑥ 칸** — `INSIDE` · `OUTSIDE` 질문을 내 주제로 바꿔 기준 점수를 다시 잽니다.
5. **⑦ 칸** — 내 주제에서 「계산」이 필요한 부분(합계 · 날짜 · 환산)이 있으면 파이썬 함수로 만듭니다.

## 제출 전 체크리스트
- [ ] 런타임 → 「세션 다시 시작 및 모두 실행」으로 처음부터 끝까지 도는지 확인했다
- [ ] 키가 코드에 적혀 있지 않다(보안 비밀만)
- [ ] 청크 설정 비교표와 「왜 이 설정인지」가 노트북에 있다
- [ ] 코랩 공유 → 「링크가 있는 모든 사용자 · 뷰어」
- [ ] 제출 글: **이름 — 챗봇 이름** / 코랩 링크 · 푸는 문제 한두 줄 · 쓴 API · 고른 청크 설정과 이유 · 화면 캡처